### Silver data transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df= spark.read.format("delta").option("header","true").option("inferschema","true").load(f"abfss://silver@netflixsharsh1.dfs.core.windows.net/netflix_titles")

In [0]:
df.display()

In [0]:
df= df.fillna({"duration_minutes":0,"duration_seasons":1})

In [0]:
df=df.withColumn("duration_minutes",col("duration_minutes").cast(IntegerType())).withColumn("duration_seasons",col("duration_seasons").cast(IntegerType()))

In [0]:
df.printSchema()

In [0]:
df=df.withColumn("shorTtitle",split(col("title"),":")[0])
df.display(5)

In [0]:
df=df.withColumn("type_flag",when(col("type")=="Movie",1).when(col("type")=="TV Show",2).otherwise(0))
display(df)

In [0]:
from pyspark.sql.window import Window
df=df.withColumn("duration_ranking",dense_rank().over(Window.orderBy(col("duration_minutes").desc())))
display(df)

In [0]:
df.createOrReplaceTempView("temp_view")
df=spark.sql("SELECT * FROM temp_view").show()

In [0]:
df.display()

In [0]:
df.createOrReplaceGlobalTempView("global_view")
df=spark.sql("SELECT * FROM global_temp.global_view")
df.display()

In [0]:
df.groupBy("type").agg(count("*").alias("total_count")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("delta").mode("overwrite")\
    .option("path","abfss://silver@netflixsharsh1.dfs.core.windows.net/netflix_titles").save()